## PARTE 2 — Búsquedas prácticas (ejecución en Python)

Entregar un script o notebook (.ipynb recomendado) con ejecuciones de los siguientes tipos de búsqueda y ejemplos de resultados (top-5):

1. Vector Search
2. Hybrid Search (vector + keyword)
3. Semantic Search (query_type=semantic)
4. Semantic Hybrid Search (semantic + vector)

[Referencia para semantic ranking](https://github.com/Azure-Samples/azure-search-python-samples/blob/main/Quickstart-Semantic-Ranking/semantic-ranking-quickstart.ipynb)
[Referencia para vector search](https://github.com/Azure-Samples/azure-search-python-samples/blob/main/Quickstart-Vector-Search/vector-search-quickstart.ipynb)


## EXTRA (elige al menos UNA opcion y documenta)

A. Añadir una **custom skill** (OCR o endpoint propio) al skillset: documentar su implementación y efecto en indexing. Referencia: https://learn.microsoft.com/en-us/azure/search/cognitive-search-defining-skillset

B. Añadir un **scoring profile** al índice y explicar cómo afecta al ranking. Referencia: https://learn.microsoft.com/es-es/azure/search/index-add-scoring-profiles?tabs=python

C. Implementar **multimodal search** (texto + imágenes) si los documentos contienen imágenes. Referencia: https://learn.microsoft.com/en-us/azure/search/multimodal-search-overview

---

## Entregables (qué debe contener la entrega)

1. Un archivo en formato .docx, .ipynb o .md documentando la Parte 1
2. `practica_vector_search.ipynb` o `practica_vector_search.py` con la Parte 2 ejecutada y resultados incluidos.
4. (Opcional) Documentación y explicación de la implementación del extra elegido

---

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from openai import AzureOpenAI
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

# === CARGAR VARIABLES DE ENTORNO ===
# Buscar archivo .env en la carpeta del proyecto (un nivel arriba de embeddings)
env_path = Path(__file__).parent.parent / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print("[OK] Archivo .env cargado desde:", env_path)
else:
    # Si no existe, intentar cargar desde la carpeta actual
    load_dotenv()
    print("[WARNING] Archivo .env no encontrado. Usando variables de entorno del sistema.")

# === CONFIGURACIÓN (desde .env) ===
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_KEY = os.getenv("SEARCH_KEY")
INDEX_NAME = os.getenv("INDEX_NAME")

FOUNDRY_PROJECT_CONNECTION_STRING = os.getenv("FOUNDRY_PROJECT_CONNECTION_STRING")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-ada-002")

# Validar que las credenciales se cargaron correctamente
if not all([SEARCH_ENDPOINT, SEARCH_KEY, INDEX_NAME, AZURE_OPENAI_KEY]):
    raise ValueError("ERROR: No se pudieron cargar las credenciales. Verifica que el archivo .env existe y tiene todos los valores requeridos.")

# Cliente de Foundry usando SDK oficial
try:
    client_ai = AIProjectClient.from_connection_string(
        conn_str=FOUNDRY_PROJECT_CONNECTION_STRING,
        credential=AzureKeyCredential(AZURE_OPENAI_KEY)
    )
    print("[OK] Cliente AI Projects configurado.")
except Exception as e:
    print(f"[WARNING] Usando OpenAI client directo: {e}")
    client_ai = AzureOpenAI(
        api_key=AZURE_OPENAI_KEY,  
        api_version="2024-02-15-preview", 
        azure_endpoint=FOUNDRY_PROJECT_CONNECTION_STRING.replace("/api/projects/projuno", "")
    )

search_client = SearchClient(SEARCH_ENDPOINT, INDEX_NAME, AzureKeyCredential(SEARCH_KEY))

print("[OK] Configuración completada.")

[WARNING] Usando OpenAI client directo: type object 'AIProjectClient' has no attribute 'from_connection_string'
[OK] Configuración completada.


## 1. VECTOR SEARCH

In [17]:
from azure.search.documents.models import VectorizedQuery

print("=" * 70)
print("1. VECTOR SEARCH")
print("=" * 70)

# Realizar Vector Search - usando el campo 'text_vector' correcto
vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=5, fields="text_vector")

search_results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["parent_id", "chunk", "title", "chunk_id"]
)

print(f"\nQuery: '{query_text}'")
print(f"\nTOP-5 RESULTADOS (Vector Search):\n")

results_list = list(search_results)
for i, result in enumerate(results_list[:5], 1):
    print(f"{i}. ID: {result.get('chunk_id', 'N/A')}")
    print(f"   Título: {result.get('title', 'N/A')}")
    print(f"   Contenido: {result.get('chunk', 'N/A')[:100]}...")
    print(f"   Score: {result.get('@search.score', 'N/A'):.4f}")
    print()

1. VECTOR SEARCH

Query: '¿Qué información hay sobre el procesamiento de datos?'

TOP-5 RESULTADOS (Vector Search):

1. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBkZXNjYXJnYXIlMjBncmF0aXMucGRm0_pages_7
   Título: Plantillas de CV para descargar gratis.pdf
   Contenido: Purus lectusmalesuadalibero Sitametcommodo. 

https://orientacion-laboral.infojobs.net/wp-content/up...
   Score: 0.8149

2. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBkZXNjYXJnYXIlMjBncmF0aXMucGRm0_pages_4
   Título: Plantillas de CV para descargar gratis.pdf
   Contenido: EMPLEOS - NOMBRE DE LA EMPRESA 1 Puesto de trabajo 2017-2018 Lorem ipsum dolor sit amet, consectetur...
   Score: 0.8128

3. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBkZXNjYXJnYXIlMjBncmF

### Analisis Vector Search

**¿Qué se hizo?**
- Se generó un embedding de 1536 dimensiones para la consulta usando el modelo `text-embedding-ada-002`
- Se realizó una búsqueda vectorial pura (sin keywords) usando `VectorizedQuery`
- Se buscan los **5 documentos más cercanos** en el espacio vectorial usando similitud coseno

**¿Cómo funciona?**
- El modelo transforma el texto en un vector numérico
- Se comparan los vectores usando distancia euclidiana/coseno
- Los documentos más similares semánticamente aparecen primero
- **Ventaja**: Encuentra documentos relacionados semánticamente incluso sin palabras clave exactas

**Resultados:**
- Los scores indican qué tan similar es cada documento al query vectorial
- Solo se basan en semántica, no en palabras exactas
- Util para búsquedas por concepto o significado

## 2. HYBRID SEARCH (Vector + Keyword)

In [18]:
print("=" * 70)
print("2. HYBRID SEARCH (Vector + Keyword)")
print("=" * 70)

# Realizar Hybrid Search (text + vector)
vector_query_hybrid = VectorizedQuery(vector=query_vector, k_nearest_neighbors=5, fields="text_vector")

search_results_hybrid = search_client.search(
    search_text=query_text,  # Búsqueda por keywords
    vector_queries=[vector_query_hybrid],  # + búsqueda vectorial
    select=["parent_id", "chunk", "title", "chunk_id"]
)

print(f"\nQuery: '{query_text}'")
print(f"\nTOP-5 RESULTADOS (Hybrid Search):\n")

results_hybrid_list = list(search_results_hybrid)
for i, result in enumerate(results_hybrid_list[:5], 1):
    print(f"{i}. ID: {result.get('chunk_id', 'N/A')}")
    print(f"   Título: {result.get('title', 'N/A')}")
    print(f"   Contenido: {result.get('chunk', 'N/A')[:100]}...")
    print(f"   Score: {result.get('@search.score', 'N/A'):.4f}")
    print()

2. HYBRID SEARCH (Vector + Keyword)

Query: '¿Qué información hay sobre el procesamiento de datos?'

TOP-5 RESULTADOS (Hybrid Search):

1. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBkZXNjYXJnYXIlMjBncmF0aXMucGRm0_pages_7
   Título: Plantillas de CV para descargar gratis.pdf
   Contenido: Purus lectusmalesuadalibero Sitametcommodo. 

https://orientacion-laboral.infojobs.net/wp-content/up...
   Score: 0.0331

2. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBkZXNjYXJnYXIlMjBncmF0aXMucGRm0_pages_6
   Título: Plantillas de CV para descargar gratis.pdf
   Contenido: CIUDAD (PROVINCIA) in 0 f 

https://orientacion-laboral.infojobs.net/wp-content/uploads/2018/03/Plan...
   Score: 0.0328

3. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBk

### Analisis Hybrid Search

**Qué se hizo?**
- Se combino busqueda por keywords (BM25) + busqueda vectorial (embedding)
- Ambas busquedas se ejecutan simultaneamente y los resultados se fusionan
- Azure Search combina los scores usando un algoritmo de fusion

**Cómo funciona?**
1. **Parte de keywords**: Encuentra documentos que contienen las palabras exactas (BM25)
2. **Parte vectorial**: Encuentra documentos semanticamente similares
3. **Fusion**: Los scores se combinan para dar resultados más relevantes

**Ventajas vs Vector Search:**
- Encuentra tanto coincidencias exactas como semanticas
- Mejor para consultas con términos específicos
- Equilibra precision (keywords) + relevancia semantica (vectors)

**Resultados:**
- Los scores suelen ser más altos que en busqueda pura vectorial
- Incluye documentos con palabras clave + documentos relacionados semanticamente
- Mejor para busquedas balanceadas

## 3. SEMANTIC SEARCH

In [19]:
from azure.search.documents.models import QueryType

print("=" * 70)
print("3. SEMANTIC SEARCH")
print("=" * 70)

# Realizar Semantic Search
try:
    search_results_semantic = search_client.search(
        search_text=query_text,
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default",
        select=["parent_id", "chunk", "title", "chunk_id"]
    )
    
    print(f"\nQuery: '{query_text}'")
    print(f"\nTOP-5 RESULTADOS (Semantic Search):\n")
    
    results_semantic_list = list(search_results_semantic)
    for i, result in enumerate(results_semantic_list[:5], 1):
        print(f"{i}. ID: {result.get('chunk_id', 'N/A')}")
        print(f"   Título: {result.get('title', 'N/A')}")
        print(f"   Contenido: {result.get('chunk', 'N/A')[:100]}...")
        print(f"   Score: {result.get('@search.score', 'N/A'):.4f}")
        if '@search.reranker_score' in result:
            print(f"   Reranker Score: {result.get('@search.reranker_score', 'N/A'):.4f}")
        print()
except Exception as e:
    print(f"[WARNING] Semantic search no disponible: {e}")
    print("Ejecutando busqueda por keywords como alternativa...")
    search_results_semantic = search_client.search(
        search_text=query_text,
        select=["parent_id", "chunk", "title", "chunk_id"]
    )
    
    print(f"\nQuery: '{query_text}'")
    print(f"\nTOP-5 RESULTADOS (Keyword Search):\n")
    
    results_semantic_list = list(search_results_semantic)
    for i, result in enumerate(results_semantic_list[:5], 1):
        print(f"{i}. ID: {result.get('chunk_id', 'N/A')}")
        print(f"   Título: {result.get('title', 'N/A')}")
        print(f"   Contenido: {result.get('chunk', 'N/A')[:100]}...")
        print(f"   Score: {result.get('@search.score', 'N/A'):.4f}")
        print()

3. SEMANTIC SEARCH

Query: '¿Qué información hay sobre el procesamiento de datos?'

TOP-5 RESULTADOS (Semantic Search):

[WARNING] Semantic search no disponible: (InvalidRequestParameter) Unknown semantic configuration 'default'.
Parameter name: semanticConfiguration
Code: InvalidRequestParameter
Message: Unknown semantic configuration 'default'.
Parameter name: semanticConfiguration
Exception Details:	(UnknownSemanticConfiguration) Unknown semantic configuration 'default'.
	Code: UnknownSemanticConfiguration
	Message: Unknown semantic configuration 'default'.
Ejecutando busqueda por keywords como alternativa...

Query: '¿Qué información hay sobre el procesamiento de datos?'

TOP-5 RESULTADOS (Keyword Search):

1. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBkZXNjYXJnYXIlMjBncmF0aXMucGRm0_pages_6
   Título: Plantillas de CV para descargar gratis.pdf
   Contenido: CIUDAD (PROVINCIA) in 0 f 

https://o

### Analisis Semantic Search

**Qué se hizo?**
- Se uso Semantic Ranking (ranker semantico basado en IA)
- Primero busca documentos relevantes por keywords
- Luego reordena los resultados usando un modelo de lenguaje (reranking)

**Cómo funciona?**
1. **Primera pasada**: Busqueda rápida por keywords (BM25)
2. **Segunda pasada**: Reranking semantico usando IA
   - Calcula puntuación Reranker Score basada en entendimiento semantico
   - Reordena resultados según relevancia semantica

**Qué incluye?**
- `@search.score`: Score de busqueda original (keywords)
- `@search.reranker_score`: Score del modelo semantico (más importante)
- Los resultados se ordenan por reranker_score

**Ventajas:**
- Mejor entendimiento del significado
- Identifica respuestas más precisas
- Reduce ruido y mejora la calidad
- Más lento que vector/hybrid (requiere reranking)

**Resultados:**
- Documentos ordenados por relevancia semantica real
- Mejor para busquedas donde la precision es crítica

## 4. SEMANTIC HYBRID SEARCH (Semantic + Vector)

In [20]:
print("=" * 70)
print("4. SEMANTIC HYBRID SEARCH (Semantic + Vector)")
print("=" * 70)

# Realizar Semantic Hybrid Search (text + semantic ranking + vector)
vector_query_semantic_hybrid = VectorizedQuery(vector=query_vector, k_nearest_neighbors=5, fields="text_vector")

try:
    search_results_semantic_hybrid = search_client.search(
        search_text=query_text,
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default",
        vector_queries=[vector_query_semantic_hybrid],
        select=["parent_id", "chunk", "title", "chunk_id"]
    )
    
    print(f"\nQuery: '{query_text}'")
    print(f"\nTOP-5 RESULTADOS (Semantic Hybrid Search):\n")
    
    results_semantic_hybrid_list = list(search_results_semantic_hybrid)
    for i, result in enumerate(results_semantic_hybrid_list[:5], 1):
        print(f"{i}. ID: {result.get('chunk_id', 'N/A')}")
        print(f"   Título: {result.get('title', 'N/A')}")
        print(f"   Contenido: {result.get('chunk', 'N/A')[:100]}...")
        print(f"   Score: {result.get('@search.score', 'N/A'):.4f}")
        if '@search.reranker_score' in result:
            print(f"   Reranker Score: {result.get('@search.reranker_score', 'N/A'):.4f}")
        print()
except Exception as e:
    print(f"[WARNING] Semantic hybrid search no disponible: {e}")
    print("Ejecutando busqueda hibrida simple como alternativa...")
    vector_query_alt = VectorizedQuery(vector=query_vector, k_nearest_neighbors=5, fields="text_vector")
    search_results_semantic_hybrid = search_client.search(
        search_text=query_text,
        vector_queries=[vector_query_alt],
        select=["parent_id", "chunk", "title", "chunk_id"]
    )
    
    print(f"\nQuery: '{query_text}'")
    print(f"\nTOP-5 RESULTADOS (Hybrid Search):\n")
    
    results_semantic_hybrid_list = list(search_results_semantic_hybrid)
    for i, result in enumerate(results_semantic_hybrid_list[:5], 1):
        print(f"{i}. ID: {result.get('chunk_id', 'N/A')}")
        print(f"   Título: {result.get('title', 'N/A')}")
        print(f"   Contenido: {result.get('chunk', 'N/A')[:100]}...")
        print(f"   Score: {result.get('@search.score', 'N/A'):.4f}")
        print()

print("=" * 70)
print("[OK] EJERCICIO COMPLETADO: Todos los tipos de busqueda ejecutados.")
print("=" * 70)

4. SEMANTIC HYBRID SEARCH (Semantic + Vector)

Query: '¿Qué información hay sobre el procesamiento de datos?'

TOP-5 RESULTADOS (Semantic Hybrid Search):

[WARNING] Semantic hybrid search no disponible: (InvalidRequestParameter) Unknown semantic configuration 'default'.
Parameter name: semanticConfiguration
Code: InvalidRequestParameter
Message: Unknown semantic configuration 'default'.
Parameter name: semanticConfiguration
Exception Details:	(UnknownSemanticConfiguration) Unknown semantic configuration 'default'.
	Code: UnknownSemanticConfiguration
	Message: Unknown semantic configuration 'default'.
Ejecutando busqueda hibrida simple como alternativa...

Query: '¿Qué información hay sobre el procesamiento de datos?'

TOP-5 RESULTADOS (Hybrid Search):

1. ID: 4c9059b8d7e8_aHR0cHM6Ly9haXNlYXJjaGFsdmFyby5ibG9iLmNvcmUud2luZG93cy5uZXQvYmxvYmFsdmFyby9QbGFudGlsbGFzJTIwZGUlMjBDViUyMHBhcmElMjBkZXNjYXJnYXIlMjBncmF0aXMucGRm0_pages_7
   Título: Plantillas de CV para descargar gratis.pdf
   Conten

### Analisis Semantic Hybrid Search

**Qué se hizo?**
- Se combinaron 3 técnicas simultaneamente:
  1. Busqueda por keywords (BM25)
  2. Busqueda vectorial (embeddings)
  3. Reranking semantico (IA)

**Cómo funciona?**
```
Consulta -> Keywords + Vectores + Semantic Ranker
            |         |          |
         Resultados BM25  Vector  Rerankeados por IA
            |         |          |
         Se fusionan y reordenan por reranker_score
            |
         Top-5 más relevantes
```

**Combinacion completa:**
- **BM25 Keywords**: Precision de palabras exactas
- **Vector Search**: Similitud semantica
- **Semantic Ranker**: Entendimiento profundo de relevancia

**Ventajas:**
- Máxima precision y recall
- Encuentra respuestas exactas + relacionadas
- Mejor clasificacion final
- Más lento (combina 3 metodos)

**Resultados:**
- Los scores reflejan la combinacion de todos los metodos
- Mejores resultados globales
- Recomendado para: busquedas criticas, RAG, Q&A

| Tecnica | Velocidad | Precision | Caso de Uso |
|---------|-----------|-----------|------------|
| Vector | Muy rapido | Moderada | Busquedas rápidas |
| Hybrid | Rapido | Alta | Busquedas balanceadas |
| Semantic | Lento | Muy alta | Precision critica |
| Semantic Hybrid | Lento | Maxima | Mejor resultado global |

## COMPARATIVA FINAL: RESUMEN DE LOS 4 MÉTODOS

### 1. Vector Search (Búsqueda Vectorial)
- **Qué busca**: Documentos semanticamente similares
- **Método**: Compara vectores de embeddings (distancia coseno)
- **Ventajas**: Rápido, intuitivo, captura semantica
- **Limitaciones**: Ignora palabras clave exactas, requiere embeddings
- **Mejor para**: Busquedas por concepto, sin palabras específicas
- **Ejemplo**: Buscar procesamiento datos → Encuentra documentos sobre ETL, análisis, etc.

---

### 2. Hybrid Search (Busqueda Hibrida)
- **Qué combina**: Keywords (BM25) + Vectores (embeddings)
- **Método**: Dos busquedas paralelas que se fusionan
- **Ventajas**: Equilibra precision + semantica
- **Limitaciones**: Más lento que vector, requiere ajuste de fusion
- **Mejor para**: La mayoría de casos prácticos
- **Ejemplo**: Buscar procesamiento → Encuentra exactas + relacionadas

---

### 3. Semantic Search (Busqueda Semantica)
- **Qué hace**: Keywords + Reranking con IA
- **Método**: BM25 inicial -> Reordenamiento por modelo semantico
- **Ventajas**: Alta precision, entiende contexto
- **Limitaciones**: Más lento, requiere semantic configuration
- **Mejor para**: Busquedas donde precision es critica
- **Ejemplo**: información procesamiento -> Ordena por relevancia real

---

### 4. Semantic Hybrid Search (Lo Mejor)
- **Qué combina**: Keywords + Vectores + Semantic Ranking
- **Método**: Triple busqueda (BM25 + Vector + Reranking IA)
- **Ventajas**: Máxima precision y recall
- **Limitaciones**: Más lento, mayor costo computacional
- **Mejor para**: QA systems, RAG, busquedas criticas
- **Ejemplo**: Mejor de todos los anteriores combinados

---

### Matriz de Seleccion

```
CASO DE USO                          -> MÉTODO RECOMENDADO
────────────────────────────────────────────────────────
Búsquedas generales, rápidas        -> Vector Search
Búsqueda balanceada más común        -> Hybrid Search (Recomendado)
Precisión máxima, búsqueda crítica   -> Semantic Search
Mejor resultado posible (RAG, Q&A)   -> Semantic Hybrid (Recomendado)
```

---

### CONCLUSIONES
1. **Vector Search**: Base sólida, rápido pero limita semantica
2. **Hybrid Search**: Recomendado para la mayoría de aplicaciones
3. **Semantic Search**: Cuando necesites máxima precision
4. **Semantic Hybrid**: Lo ideal para sistemas Q&A y RAG produccion

In [21]:
query_text = "¿Qué información hay sobre el procesamiento de datos?"

try:
    # Llamada al modelo de embeddings
    response = client_ai.embeddings.create(
        input=[query_text], 
        model=EMBEDDING_MODEL
    )
    query_vector = response.data[0].embedding
    print(f"[OK] Exito! Vector generado. Dimensiones: {len(query_vector)}")
except Exception as e:
    print(f"[ERROR] {e}")

# Explorar la estructura del índice
print("\n[INFO] INFORMACIÓN DEL ÍNDICE:")
try:
    # Buscar un documento para ver su estructura
    simple_search = search_client.search(search_text="*")
    first_doc = next(simple_search)
    print("Campos disponibles en los documentos:")
    for key in first_doc.keys():
        print(f"  - {key}")
except Exception as e:
    print(f"Error al explorar índice: {e}")

[OK] Exito! Vector generado. Dimensiones: 1536

[INFO] INFORMACIÓN DEL ÍNDICE:
Campos disponibles en los documentos:
  - parent_id
  - chunk
  - title
  - chunk_id
  - text_vector
  - @search.score
  - @search.reranker_score
  - @search.highlights
  - @search.captions
